# 🎵 Latin Music Feature Dataset Generator

This notebook builds a machine-learning-ready audio feature dataset from a Latin music playlist.

For each song it:
1. Searches YouTube for the best official audio source (prefers *Topic* channels)
2. Downloads the full audio
3. Extracts up to **N audio fragments** per song, chosen from the most-replayed moments (YouTube heatmap)
4. Computes **CLAP audio embeddings** (512 dims) + acoustic features (BPM, 13 MFCCs) for every fragment

The result is a CSV file where each row is one fragment, ready for genre classification.

---

### Key features
| Feature | Description |
|---|---|
| **Resumable** | Already-processed songs are skipped; interrupted runs continue where they left off |
| **Multi-fragment** | Extracts multiple diverse fragments per song to improve classifier generalisation |
| **Smart search** | Topic channel → Official audio → Simple search fallback |
| **Group-aware** | `song_id` column enables group-aware train/test splits (no data leakage) |
| **Changelog** | v2: Topic-channel search, full-audio download, heatmap-based fragment selection, `song_id` column |


## 1 · Install Dependencies

Run this cell once per Kaggle session. It installs:
- **Node.js 20** — required by yt-dlp to solve YouTube's JavaScript challenges
- **yt-dlp** — the YouTube downloader


In [1]:
# Install Node.js 20 (needed by yt-dlp for JS challenge solving)
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs

# Install / upgrade yt-dlp
!pip install -q -U yt-dlp


2026-05-05 18:09:07 - Installing pre-requisites
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [90.8 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,608 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,995 kB]
Get:13 https:/

## 2 · Hugging Face Authentication

The CLAP model (`laion/clap-htsat-unfused`) is a public model, but logging in avoids rate-limiting.

**Setup:** Add a Kaggle secret named `HF_TOKEN` with your Hugging Face access token.  
Generate one at https://huggingface.co/settings/tokens (read-only scope is sufficient).


In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)


## 3 · Imports & Configuration

**Paths to review before running:**
- `CSV_PATH` — your input playlist CSV (must have `track` and `genre` columns)
- `OUTPUT_CSV` — where the feature dataset will be saved
- `PROCESSED_CSV` — (optional) a previously generated CSV to resume from
- `COOKIES_SRC` — (optional) YouTube cookies file to avoid bot-detection

**Tunable parameters:**
| Parameter | Default | Description |
|---|---|---|
| `MAX_FRAGMENTS` | 3 | Fragments extracted per song. Use 1 for short songs (<90s) |
| `MIN_FRAGMENT_DISTANCE_SEC` | 45 | Minimum gap (seconds) between fragment start times |
| `FRAGMENT_DURATION_SEC` | 30 | Duration of each extracted fragment |


In [3]:
import os
import uuid
import shutil
import glob
import subprocess

import torch
import librosa
import numpy as np
import pandas as pd
from transformers import ClapModel, ClapProcessor

# Make sure /usr/bin is in PATH (required to find the node binary)
os.environ['PATH'] = '/usr/bin:' + os.environ.get('PATH', '')

# ── File paths ────────────────────────────────────────────────────────────────
# Input playlist CSV — must contain at least 'track' and 'genre' columns
CSV_PATH       = '/kaggle/input/datasets/g5fighter/playlist-latin-music/spotify_playlists_dataset.csv'

# Output: where the generated feature dataset is saved
OUTPUT_CSV     = '/kaggle/working/latin_music_features_dataset.csv'

# Optional: a previously generated CSV to resume from
# If OUTPUT_CSV doesn't exist yet but this file does, it will be copied over
PROCESSED_CSV  = '/kaggle/input/datasets/g5fighter/latin-music-embeddings/latin_music_features_dataset.csv'

# Temporary directory for downloaded audio files (cleaned up after each song)
TEMP_AUDIO_DIR = '/kaggle/working/temp_audio'

# ── YouTube cookies (optional but recommended) ────────────────────────────────
# yt-dlp needs write access to the cookie file (it refreshes tokens),
# so it cannot live in /kaggle/input/ (read-only). We copy it to /kaggle/working/.
_COOKIES_SRC = '/kaggle/input/datasets/g5fighter/youtube-cookies/www.youtube.com_cookies.txt'
COOKIES_FILE = '/kaggle/working/youtube_cookies.txt'

if os.path.exists(_COOKIES_SRC):
    shutil.copy(_COOKIES_SRC, COOKIES_FILE)
    print(f'✅ Cookies copied to {COOKIES_FILE}')
else:
    COOKIES_FILE = None
    print('⚠️  Cookie file not found — continuing without YouTube authentication')

# ── Fragment extraction parameters ───────────────────────────────────────────
# Number of fragments to extract per song.
#   1  → original behaviour (single fragment at the heatmap peak)
#   2-4 → multiple fragments spread across the song for richer diversity
# If the heatmap doesn't have enough spread-out peaks, fewer fragments are
# extracted without raising an error.  For songs shorter than ~90s use 1.
MAX_FRAGMENTS = 3

# Minimum gap (in seconds) between fragment start times.
# 45s means two 30s fragments share no audio → more varied embeddings.
MIN_FRAGMENT_DISTANCE_SEC = 45

# Duration (in seconds) of each extracted audio fragment
FRAGMENT_DURATION_SEC = 30

os.makedirs(TEMP_AUDIO_DIR, exist_ok=True)

# ── Load the CLAP model ───────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

print('Loading CLAP model (laion/clap-htsat-unfused)…')
MODEL_ID  = 'laion/clap-htsat-unfused'
processor = ClapProcessor.from_pretrained(MODEL_ID)
model     = ClapModel.from_pretrained(MODEL_ID)

if torch.cuda.device_count() > 1:
    print(f'Enabling DataParallel across {torch.cuda.device_count()} GPUs')
    model = torch.nn.DataParallel(model)

model = model.to(device)
model.eval()
print('✅ CLAP model ready.')


✅ Cookies copied to /kaggle/working/youtube_cookies.txt
Using device: cuda
Loading CLAP model (laion/clap-htsat-unfused)…


preprocessor_config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/615M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/614M [00:00<?, ?B/s]

Enabling DataParallel across 2 GPUs
✅ CLAP model ready.


## 4 · Pre-fetch EJS Runtime

yt-dlp uses an EJS script from GitHub to solve YouTube's JavaScript challenges.  
This cell checks that the runtime is available before the main loop starts.


In [4]:
import yt_dlp

print('Verifying yt-dlp EJS runtime…')
result = subprocess.run(
    [
        'yt-dlp',
        '--js-runtimes', 'node:/usr/bin/node',
        '--remote-components', 'ejs:github',
        '--list-formats',
        'https://www.youtube.com/watch?v=GGcz4Sr3s74',
    ],
    capture_output=True,
    text=True,
    env={**os.environ, 'PATH': '/usr/bin:' + os.environ.get('PATH', '')},
)

if 'm4a' in result.stdout or 'webm' in result.stdout:
    print('✅ EJS runtime ready')
else:
    print('⚠️  EJS runtime may not have loaded correctly — downloads could fail')
    print(result.stderr[-500:])


Verifying yt-dlp EJS runtime…
✅ EJS runtime ready


## 5 · Download & Fragment-Selection Helpers

Three functions are defined here:

| Function | Purpose |
|---|---|
| `_ydl_info_opts()` | Returns yt-dlp options shared by the metadata and download calls |
| `search_official_video(query)` | Searches YouTube with a 3-tier fallback strategy |
| `select_fragment_times(...)` | Picks N start times from the YouTube heatmap (or falls back to uniform spacing) |
| `download_full_audio(url, song_id)` | Downloads the complete audio as WAV |


In [5]:
def _ydl_info_opts() -> dict:
    """Common yt-dlp options for metadata extraction (no download)."""
    opts = {
        'quiet': True,
        'no_warnings': True,
        'noplaylist': True,
        'nocheckcertificate': True,
        'format': 'bestaudio/best',
        'extractor_args': {
            'youtube': {
                'player_client': ['web'],
                'player_skip': ['webpage'],
            }
        },
        'js_runtimes': {'node': {'path': '/usr/bin/node'}},
        'remote_components': {'ejs': 'github'},
    }
    if COOKIES_FILE and os.path.exists(COOKIES_FILE):
        opts['cookiefile'] = COOKIES_FILE
    return opts


def search_official_video(query: str, expected_artist: str = None) -> dict | None:
    """
    Find the best YouTube video for a song using a 3-tier search strategy:

    1. **Topic channel**  — appends '- Topic' and validates the artist name.
    2. **Official audio** — appends 'official audio' and validates uploader/title.
    3. **Plain search**   — uses raw query and validates uploader/title.

    Parameters
    ----------
    query           : Search terms (usually track name).
    expected_artist : Name of the artist to validate against uploader/channel.

    Returns the yt-dlp info dict for the chosen video, or None if all tiers fail.
    """
    artist_clean = expected_artist.lower().strip() if expected_artist else ""
    
    strategies = [
        (f'ytsearch5:{query} - Topic',       'Topic channel'),
        (f'ytsearch5:{query} official audio', 'Official audio'),
        (f'ytsearch5:{query}',                'Plain search'),
    ]

    with yt_dlp.YoutubeDL(_ydl_info_opts()) as ydl:
        for search_query, strategy_name in strategies:
            try:
                info    = ydl.extract_info(search_query, download=False)
                entries = info.get('entries', [])

                if not entries:
                    continue

                for entry in entries:
                    uploader = (entry.get('uploader') or "").lower()
                    channel  = (entry.get('channel') or "").lower()
                    title    = (entry.get('title') or "").lower()

                    # 1. Validation for Topic Channels: Must end in '- Topic' and contain artist
                    if strategy_name == 'Topic channel':
                        if channel.strip().endswith('- topic') and artist_clean in channel:
                            print(f'   -> Topic channel match: {entry.get("channel")}')
                            return entry
                    
                    # 2. Validation for Fallbacks: Artist must be in uploader or title
                    else:
                        # Exclude media/news channels (e.g., "SIN FILTROS TV") unless they are the artist
                        blacklist = [' tv', 'news', 'interview', 'entrevista', 'show']
                        if any(word in uploader for word in blacklist) and artist_clean not in uploader:
                            continue
                            
                        if artist_clean in uploader or artist_clean in title:
                            print(f'   -> Using fallback ({strategy_name}): {entry.get("uploader")}')
                            return entry

            except Exception as e:
                print(f'   -> Error in strategy "{strategy_name}": {e}')

    return None  # All strategies exhausted


def select_fragment_times(
    heatmap: list | None,
    total_duration: float,
    n_fragments: int,
    fragment_duration: float,
    min_distance: float,
) -> list[float]:
    """
    Choose N start times for audio fragments.

    With a heatmap:
      - Ranks all segments by replay value (descending)
      - Greedily picks times that are at least `min_distance` apart
      - Stops when N fragments are selected or candidates run out

    Without a heatmap (fallback):
      - Distributes fragments uniformly between the 10 % and 90 % marks
        (skips intro and outro)

    Parameters
    ----------
    heatmap          : list of dicts with 'start_time' and 'value' from yt-dlp
    total_duration   : total song length in seconds
    n_fragments      : maximum number of fragments to return
    fragment_duration: length of each fragment in seconds
    min_distance     : minimum gap between fragment start times in seconds

    Returns a list of start times (floats). May be shorter than n_fragments
    if the song is too short or the heatmap lacks enough spread-out peaks.
    """
    max_start = total_duration - fragment_duration
    if max_start <= 0:
        return [0.0]

    if heatmap:
        # Sort by replay value — highest first
        candidates = sorted(heatmap, key=lambda x: x.get('value', 0), reverse=True)
        selected   = []

        for segment in candidates:
            t = float(segment.get('start_time', 0))
            t = max(0.0, min(t, max_start))  # clamp to valid range

            # Skip if too close to an already-selected time
            too_close = any(abs(t - t_prev) < min_distance for t_prev in selected)
            if not too_close:
                selected.append(t)

            if len(selected) == n_fragments:
                break

        if selected:
            return selected

    # ── Fallback: uniform spacing between 10 % and 90 % of the song ──
    zone_start = total_duration * 0.10
    zone_end   = max(total_duration * 0.90 - fragment_duration, zone_start)

    if n_fragments == 1:
        return [zone_start + (zone_end - zone_start) / 2]

    step   = (zone_end - zone_start) / (n_fragments - 1)
    times  = [zone_start + i * step for i in range(n_fragments)]

    # Drop any times that ended up too close together after clamping
    filtered = [times[0]]
    for t in times[1:]:
        if t - filtered[-1] >= min_distance:
            filtered.append(t)
    return filtered


def download_full_audio(url: str, song_id: str) -> str | None:
    """
    Download the full audio from a YouTube URL and convert it to WAV.

    The full audio is downloaded (not just a 30-second clip) so that multiple
    fragments at different timestamps can be extracted without re-downloading.

    Parameters
    ----------
    url     : YouTube video URL
    song_id : unique identifier used as the output filename stem

    Returns the path to the WAV file, or None if the download failed.
    """
    out_template = os.path.join(TEMP_AUDIO_DIR, f'{song_id}.%(ext)s')
    wav_path     = os.path.join(TEMP_AUDIO_DIR, f'{song_id}.wav')

    # Remove any leftover file from a previous attempt
    if os.path.exists(wav_path):
        os.remove(wav_path)

    ydl_opts = {
        'format': 'bestaudio[ext=webm]/bestaudio[ext=m4a]/bestaudio/best',
        'outtmpl': out_template,
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'quiet': True,
        'no_warnings': True,
        'nocheckcertificate': True,
        'noplaylist': True,
        'restrictfilenames': True,
        'extractor_args': {
            'youtube': {
                'player_client': ['web'],
                'player_skip': ['webpage'],
            }
        },
        'js_runtimes': {'node': {'path': '/usr/bin/node'}},
        'remote_components': {'ejs': 'github'},
    }
    if COOKIES_FILE and os.path.exists(COOKIES_FILE):
        ydl_opts['cookiefile'] = COOKIES_FILE

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        files = glob.glob(os.path.join(TEMP_AUDIO_DIR, f'{song_id}.*'))
        return files[0] if files else None
    except Exception as e:
        print(f'   -> Download error: {e}')
        return None


print('✅ Download and fragment-selection helpers defined.')


✅ Download and fragment-selection helpers defined.


## 6 · Per-Fragment Feature Extraction

`extract_fragment_features` loads a slice of the WAV file and returns:

- **BPM** — beats per minute (librosa beat tracker)
- **MFCCs** — 13 Mel-frequency cepstral coefficient means (`mfcc_0` … `mfcc_12`)
- **CLAP embedding** — 512-dimensional audio embedding (`emb_0` … `emb_511`)
- **fragment_start_sec** — the start time used (stored for reproducibility)

Fragments are silently discarded if they are shorter than 5 seconds, silent, or produce NaN embeddings.


In [6]:
def extract_fragment_features(
    audio_path: str,
    start_sec: float,
    duration: float = FRAGMENT_DURATION_SEC,
) -> dict | None:
    """
    Compute CLAP embeddings + acoustic features for a single audio fragment.

    Parameters
    ----------
    audio_path : path to the full WAV file
    start_sec  : fragment start time in seconds
    duration   : fragment length in seconds (default: FRAGMENT_DURATION_SEC)

    Returns a dict with keys 'bpm', 'fragment_start_sec', 'mfcc_0'–'mfcc_12',
    'emb_0'–'emb_511', or None if the fragment is invalid.
    """
    try:
        # Load only the needed slice — more efficient than loading the whole file
        y_clap, _ = librosa.load(audio_path, sr=48_000, offset=start_sec, duration=duration)

        # Reject fragments that are too short (< 5 seconds)
        if len(y_clap) < 48_000 * 5:
            print(f'   -> Fragment at {start_sec:.1f}s is too short — skipping.')
            return None

        # Reject silent fragments
        if np.max(np.abs(y_clap)) < 1e-4:
            print(f'   -> Fragment at {start_sec:.1f}s is silent — skipping.')
            return None

        # Resample to 22 050 Hz for librosa acoustic features
        y_features = librosa.resample(y_clap, orig_sr=48_000, target_sr=22_050)
        sr_features = 22_050

        # ── CLAP embedding (512 dimensions) ──────────────────────────────────
        inputs = processor(audio=y_clap, sampling_rate=48_000, return_tensors='pt').to(device)
        with torch.no_grad():
            actual_model  = model.module if hasattr(model, 'module') else model
            audio_features = actual_model.get_audio_features(**inputs)
            embedding      = audio_features[0].cpu().numpy().flatten()

        if np.isnan(embedding).any():
            print(f'   -> Embedding contains NaNs at {start_sec:.1f}s — skipping.')
            return None

        # ── BPM + MFCCs ───────────────────────────────────────────────────────
        tempo, _   = librosa.beat.beat_track(y=y_features, sr=sr_features)
        mfccs      = librosa.feature.mfcc(y=y_features, sr=sr_features, n_mfcc=13)
        mfcc_means = np.mean(mfccs, axis=1)
        bpm        = float(np.mean(tempo))

        record = {'bpm': bpm, 'fragment_start_sec': start_sec}
        for i in range(13):
            record[f'mfcc_{i}'] = mfcc_means[i]
        for i in range(512):
            record[f'emb_{i}'] = embedding[i]

        return record

    except Exception as e:
        print(f'   -> Error processing fragment at {start_sec:.1f}s: {e}')
        return None


print('✅ Feature extraction function defined.')


✅ Feature extraction function defined.


## 7 · Load Input CSV & Build Resume State

This cell:
1. Loads the input playlist CSV
2. Copies `PROCESSED_CSV` to `OUTPUT_CSV` if there is no existing output yet (resume from a previous Kaggle run)
3. Reads the existing output CSV (if any) to build an in-memory state that tracks which fragments have already been extracted for each song


In [7]:
import unicodedata
import pandas as pd
import os
import uuid
import shutil

# ── 1. Load input playlist ────────────────────────────────────────────────────
try:
    df_input = pd.read_csv(CSV_PATH)
    print(f'Input CSV loaded: {len(df_input)} songs.')
    valid_tracks = set(df_input['track'].tolist())
except Exception as e:
    print(f'Error loading input CSV: {e}')
    df_input      = pd.DataFrame()
    valid_tracks  = set()

# ── 2. Resume from a previous run ────────────────────────────────────────────
song_state: dict = {}

if not os.path.exists(OUTPUT_CSV) and os.path.exists(PROCESSED_CSV):
    shutil.copy(PROCESSED_CSV, OUTPUT_CSV)
    print('✅ Resumed from PROCESSED_CSV.')

if os.path.exists(OUTPUT_CSV):
    try:
        df_existing      = pd.read_csv(OUTPUT_CSV)
        n_songs_before   = df_existing['track'].nunique()

        # Drop rows whose songs are no longer in the input playlist
        df_existing      = df_existing[df_existing['track'].isin(valid_tracks)].copy()
        n_songs_removed  = n_songs_before - df_existing['track'].nunique()

        # Sync genre labels with the current input CSV
        genre_map            = dict(zip(df_input['track'], df_input['genre']))
        df_existing['genre'] = df_existing['track'].map(genre_map)

        if 'fragment_index' not in df_existing.columns:
            df_existing['fragment_index'] = 0

        if 'fragment_start_sec' not in df_existing.columns:
            df_existing['fragment_start_sec'] = 0.0

        df_existing.to_csv(OUTPUT_CSV, index=False)
        print(f'🧹 Removed {n_songs_removed} obsolete songs. Genre labels refreshed.')

        # Build per-song state
        for track, group in df_existing.groupby('track'):
            song_state[track] = {
                'song_id' : group['song_id'].iloc[0],
                'indices' : set(group['fragment_index'].tolist()),
                'times'   : dict(zip(group['fragment_index'], group['fragment_start_sec'])),
            }

        n_complete   = sum(1 for s in song_state.values() if len(s['indices']) >= MAX_FRAGMENTS)
        n_incomplete = len(song_state) - n_complete
        print(f'Resume state: {n_complete} complete songs, {n_incomplete} with missing fragments.')

    except Exception as e:
        print(f'Error reading existing output CSV: {e}')
else:
    print('No existing output CSV found — starting fresh.')

Input CSV loaded: 2209 songs.
✅ Resumed from PROCESSED_CSV.
🧹 Removed 0 obsolete songs. Genre labels refreshed.
Resume state: 1988 complete songs, 28 with missing fragments.


## 8 · Main Extraction Loop

Iterates through every song in the input CSV and:

1. **Skips** songs where all `MAX_FRAGMENTS` fragments are already saved
2. **Searches YouTube** for the best official audio source
3. **Selects fragment times** from the YouTube replay heatmap
4. **Downloads** the full audio
5. **Extracts features** only for missing fragment indices
6. **Appends** new rows to the output CSV immediately (safe against interruptions)

The loop aborts automatically after `MAX_CONSECUTIVE_FAILURES` consecutive failures.


In [8]:
MAX_CONSECUTIVE_FAILURES = 40  # abort after this many back-to-back failures

consecutive_failures = 0

for row_idx, row in df_input.iterrows():
    genre = row['genre']
    track = row['track']
    state = song_state.get(track)

    # ── Determine which fragment indices are still missing ────────────────────
    existing_indices = state['indices'] if state else set()
    target_indices   = set(range(MAX_FRAGMENTS))
    missing_indices  = target_indices - existing_indices

    if not missing_indices:
        continue  # this song is fully processed

    is_new  = state is None
    song_id = state['song_id'] if state else str(uuid.uuid4())
    saved_times = state['times'] if state else {}

    label = 'NEW' if is_new else f'INCOMPLETE (missing indices: {sorted(missing_indices)})'
    print(f'\n[{row_idx + 1}/{len(df_input)}] {label}: {track} ({genre})')

    # ── Step 1: Search for official video ─────────────────────────────────────
    video_info = search_official_video(track, row.get('artist'))
    if video_info is None:
        print('   -> No video found — skipping.')
        consecutive_failures += 1
        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            print(f'🛑 {MAX_CONSECUTIVE_FAILURES} consecutive failures — aborting.')
            break
        continue

    total_duration = float(video_info.get('duration', 0) or 0)
    heatmap        = video_info.get('heatmap')
    video_url      = video_info.get('webpage_url', '')

    # ── Step 2: Select fragment start times ───────────────────────────────────
    # Always recompute from the heatmap for consistency, but preserve already-saved
    # timestamps for indices that were previously extracted successfully.
    all_times = select_fragment_times(
        heatmap=heatmap,
        total_duration=total_duration,
        n_fragments=MAX_FRAGMENTS,
        fragment_duration=FRAGMENT_DURATION_SEC,
        min_distance=MIN_FRAGMENT_DISTANCE_SEC,
    )

    # Build a full index → time map:  prefer saved times for existing indices
    time_map = {}
    for idx, t in enumerate(all_times):
        time_map[idx] = saved_times.get(idx, t)  # use saved time if available

    times_to_extract = {idx: time_map[idx] for idx in missing_indices if idx in time_map}

    if not times_to_extract:
        print('   -> No valid times for missing fragments — skipping.')
        continue

    print(f'   -> Extracting {len(times_to_extract)} fragment(s): '
          + ', '.join(f'idx={i} t={t:.1f}s' for i, t in sorted(times_to_extract.items())))

    # ── Step 3: Download full audio ───────────────────────────────────────────
    audio_path = download_full_audio(video_url, song_id)
    if audio_path is None:
        print('   -> Download failed — skipping.')
        consecutive_failures += 1
        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            print(f'🛑 {MAX_CONSECUTIVE_FAILURES} consecutive failures — aborting.')
            break
        continue

    # ── Step 4: Extract features for each missing fragment ────────────────────
    n_saved = 0
    try:
        for frag_idx, start_sec in sorted(times_to_extract.items()):
            print(f'   -> Fragment idx={frag_idx} (t={start_sec:.1f}s)…')
            features = extract_fragment_features(audio_path, start_sec)

            if features is None:
                continue

            row_record = {
                'track'              : track,
                'genre'              : genre,
                'song_id'            : song_id,
                'fragment_index'     : frag_idx,
                **features,          # includes fragment_start_sec, bpm, mfcc_*, emb_*
            }

            pd.DataFrame([row_record]).to_csv(
                OUTPUT_CSV,
                mode='a',
                header=not os.path.exists(OUTPUT_CSV),
                index=False,
            )
            n_saved += 1

            # Update in-memory state so this run doesn't re-process this index
            if track not in song_state:
                song_state[track] = {'song_id': song_id, 'indices': set(), 'times': {}}
            song_state[track]['indices'].add(frag_idx)
            song_state[track]['times'][frag_idx] = start_sec

        if n_saved > 0:
            consecutive_failures = 0
            total_for_song = len(song_state.get(track, {}).get('indices', set()))
            print(f'   ✅ {n_saved} new fragment(s) saved. Total for this song: {total_for_song}/{MAX_FRAGMENTS}.')
        else:
            print('   -> No valid fragments extracted.')
            consecutive_failures += 1

    except Exception as e:
        print(f'   -> Unexpected error: {e}')
        consecutive_failures += 1

    finally:
        # Always clean up the temporary audio file
        if audio_path and os.path.exists(audio_path):
            os.remove(audio_path)

    if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
        print(f'🛑 {MAX_CONSECUTIVE_FAILURES} consecutive failures — aborting.')
        break

print('\n✅ Extraction complete.')



[34/2209] INCOMPLETE (missing indices: [2]): Carisimo (Reggaeton)
   -> Topic channel match: Menaay - Topic
   -> No valid times for missing fragments — skipping.

[134/2209] INCOMPLETE (missing indices: [2]): Boté El Celular (Reggaeton)
   -> Using fallback (Official audio): Marc Magán
   -> No valid times for missing fragments — skipping.

[179/2209] INCOMPLETE (missing indices: [2]): Como Yai [ Remix ] (Reggaeton)
   -> Topic channel match: Najove - Topic
   -> No valid times for missing fragments — skipping.

[299/2209] INCOMPLETE (missing indices: [2]): Ouyaya (Reggaeton)
   -> Topic channel match: Najove - Topic
   -> No valid times for missing fragments — skipping.

[306/2209] INCOMPLETE (missing indices: [2]): Amorfoda (Reggaeton)
   -> Using fallback (Official audio): Audio - topic
   -> No valid times for missing fragments — skipping.

[336/2209] INCOMPLETE (missing indices: [2]): NOTELOVOYANEGAR (Reggaeton)
   -> Topic channel match: NAEL - Topic
   -> No valid times for mi

## 9 · Dataset Summary

Displays a quick overview of the generated CSV.


In [9]:
if os.path.exists(OUTPUT_CSV):
    df_out = pd.read_csv(OUTPUT_CSV)
    print(f'Total rows (fragments)  : {len(df_out)}')
    print(f'Unique songs            : {df_out["song_id"].nunique()}')
    print(f'\nFragments-per-song stats:')
    print(df_out.groupby('song_id').size().describe().to_string())
    print('\nBreakdown by genre:')
    print(
        df_out.groupby('genre').agg(
            songs=('song_id', 'nunique'),
            fragments=('song_id', 'count'),
        ).to_string()
    )
else:
    print('Output CSV does not exist yet.')


Total rows (fragments)  : 6163
Unique songs            : 2065

Fragments-per-song stats:
count    2065.000000
mean        2.984504
std         0.131155
min         1.000000
25%         3.000000
50%         3.000000
75%         3.000000
max         3.000000

Breakdown by genre:
           songs  fragments
genre                      
Bachata      914       2726
Merengue     131        390
Reggaeton    680       2027
Salsa        340       1020
